# 03 · Writing a custom PlatformAdapter

Oransim's platform layer is designed as a two-axis extension: `PlatformAdapter` × `DataProvider`. This notebook walks through subclassing `PlatformAdapter` for an imagined platform ('RedShelf') backed by a `SyntheticProvider` — enough to plug a new platform into the prediction pipeline.

In [ ]:
import sys

sys.path.insert(0, '../backend')

from oransim.platforms.base import PlatformAdapter

# Inspect the abstract interface
help(PlatformAdapter)

In [ ]:
import random
from dataclasses import dataclass
from typing import Any


# A minimal synthetic data provider for the new platform
class RedShelfSyntheticProvider:
    def __init__(self, seed: int = 42):
        self.rng = random.Random(seed)

    def fetch_kol(self, kol_id: str) -> dict:
        return {
            'kol_id': kol_id,
            'nickname': f'BookNerd_{kol_id[-4:]}',
            'fan_count': self.rng.randint(5_000, 500_000),
            'avg_engagement': round(self.rng.uniform(0.02, 0.08), 3),
        }


# A minimal RedShelf platform adapter
@dataclass
class RedShelfAdapter(PlatformAdapter):
    platform_id: str = 'redshelf'
    data_provider: Any = None

    def simulate_impression(self, creative: Any, budget: float, **kwargs) -> dict:
        # Simplified Hill-saturation impression model
        K = 50_000.0
        ratio = budget / K
        effective = (1.0 + 1.0) * ratio / (1.0 + ratio)
        return {
            'impressions': budget * 0.004 * effective,  # redshelf is text-heavy → higher CPM
            'platform': self.platform_id,
        }

    def simulate_conversion(self, impression: Any, **kwargs) -> dict:
        return {'conversions': impression['impressions'] * 0.012}

    def get_kol(self, kol_id: str) -> dict:
        return self.data_provider.fetch_kol(kol_id)

In [ ]:
# Instantiate and use the new adapter
adapter = RedShelfAdapter(data_provider=RedShelfSyntheticProvider())

imp = adapter.simulate_impression(creative={'caption':'book review'}, budget=60_000)
conv = adapter.simulate_conversion(imp)
kol = adapter.get_kol('KOL_00042')

print(f"impressions: {imp['impressions']:.0f}")
print(f"conversions: {conv['conversions']:.0f}")
print(f"sample KOL:  {kol}")

That's the minimum. A production adapter would also:

1. **Return canonical types** from `oransim.data.schema` (CanonicalKOL, CanonicalNote, CanonicalFanProfile) — v0.2 will formalize these Pydantic models.
2. **Accept multiple DataProviders** (Synthetic, CSV, JSON, OpenAPI) so the same adapter logic works across data sources.
3. **Implement the full 14–19 schema output** by wiring into `oransim.agents.schema_outputs`.

Submit a new platform adapter as a PR — see `docs/en/platforms/writing-an-adapter.md`.